In [57]:
import pandas as pd

In [79]:
df = pd.read_csv('./csv_input/07282159_modified_shapefile.csv')
community_matrix = pd.read_csv('./csv_input/Info_CSV/community_matrix.csv')
df_analysis = df.copy()

region_map = {
    'Alma Valley East': 'Capital',
    'Alma Valley West': 'Capital',
    'Highland': 'Highland',
    'Midlands': 'Lowland',
    'South East': 'Lowland',
}
df_analysis['Matrix_Region'] = df_analysis['Region'].map(region_map)

pop_type_cols = ['S', 'A1', 'A2', 'B1', 'B2', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2', 'F1', 'F2']

cm = community_matrix[['Region', 'Type', 'Code'] + pop_type_cols].rename(
    columns={'Region': 'CM_Region', 'Type': 'CM_Type', 'Code': 'CM_Code'}
)

merged = df_analysis.merge(
    cm,
    left_on=['Matrix_Region', 'Location', 'County_Type'],
    right_on=['CM_Region', 'CM_Type', 'CM_Code'],
    how='left',
)

# convert each row's per-code percentage into an actual population count
for col in pop_type_cols:
    merged[col] = merged[col] / 100.0 * merged['Population']

county_population = merged.groupby('County').agg(
    Zone=('Zone', 'first'),
    **{col: (col, 'sum') for col in pop_type_cols}
).reset_index()

county_population[pop_type_cols] = county_population[pop_type_cols].round(0).astype(int)
county_population = county_population[['County', 'Zone'] + pop_type_cols]

zone_region = df_analysis[['Zone', 'Region']].drop_duplicates().set_index('Zone')['Region']

zone_population = county_population.groupby('Zone')[pop_type_cols].sum().reset_index()
zone_population.insert(1, 'Region', zone_population['Zone'].map(zone_region))
zone_population = zone_population.sort_values('Region').reset_index(drop=True)

with pd.ExcelWriter('./analysis/population_by_type.xlsx') as writer:
    county_population.to_excel(writer, sheet_name='County', index=False)
    zone_population.to_excel(writer, sheet_name='Zone', index=False)

zone_population

,Zone,Region,S,A1,A2,B1,B2,C1,C2,D1,D2,E1,E2,F1,F2
0,Alington,Alma Valley East,144671,178455,188608,694513,117980,266298,236525,672770,90529,301677,164343,283338,50444
1,Twicken Meads,Alma Valley East,22169,56996,83903,316881,52630,164058,130904,509102,52811,218012,104598,256294,49152
2,Tedding Temple,Alma Valley East,21413,33423,99632,282782,61882,233307,185421,673693,67449,313138,130697,284248,61592
3,Techno Garden,Alma Valley East,50162,68556,53382,1619757,129247,66028,62009,112004,41347,52228,214597,44752,31044
4,Leighton,Alma Valley East,461,1043,1833,4382,1824,7187,4588,24974,989,18779,4193,25895,13131
5,Gorton Quarters,Alma Valley East,7687,14454,51778,166280,33171,130976,90086,413185,28560,225597,88295,246526,62398
6,Central,Alma Valley East,21389,46418,94166,551916,125108,160224,131428,420688,55076,195272,108332,185445,47788
7,Durdham Mews,Alma Valley East,13524,20683,35196,164545,25520,86148,66559,232579,21438,120058,46683,126027,30087
8,Denton Town,Alma Valley East,4755,11742,22490,129107,19481,56663,42906,187536,16340,81855,34513,86861,38707
9,Permbroke Corner,Alma Valley West,3552,9172,9904,38892,7239,34028,29792,70472,5189,49805,17662,69619,70677


,Zone,Region,S,A1,A2,B1,B2,C1,C2,D1,D2,E1,E2,F1,F2
0,Alington,Alma Valley East,144666,178454,188597,694441,117967,266277,236514,672494,90503,301606,164317,283323,50433
1,Twicken Meads,Alma Valley East,22169,56996,83903,316881,52630,164058,130904,509102,52811,218012,104598,256294,49152
2,Tedding Temple,Alma Valley East,21413,33423,99632,282782,61882,233307,185421,673693,67449,313138,130697,284248,61592
3,Techno Garden,Alma Valley East,50162,68556,53382,1619757,129247,66028,62009,112004,41347,52228,214597,44752,31044
4,Leighton,Alma Valley East,461,1043,1833,4382,1824,7187,4588,24974,989,18779,4193,25895,13131
5,Gorton Quarters,Alma Valley East,7687,14454,51778,166280,33171,130976,90086,413185,28560,225597,88295,246526,62398
6,Central,Alma Valley East,21389,46418,94166,551916,125108,160224,131428,420688,55076,195272,108332,185445,47788
7,Durdham Mews,Alma Valley East,13524,20683,35196,164545,25520,86148,66559,232579,21438,120058,46683,126027,30087
8,Denton Town,Alma Valley East,4755,11742,22490,129107,19481,56663,42906,187536,16340,81855,34513,86861,38707
9,Permbroke Corner,Alma Valley West,3552,9172,9904,38892,7239,34028,29792,70472,5189,49805,17662,69619,70677
